# Lab 22-06 · Switch Losses

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import losses as ls


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The hard turn-on

*Predict* Q_oss, E_oss, Q_oss·V and 2E_oss at 400 V for C(v) = 2 nF/(1 + v/2 V)^0.9, and which one the channel dissipates.

In [2]:
q = ls.qoss_junction(2e-9, 2, 0.9, 400)
e = ls.eoss_junction(2e-9, 2, 0.9, 400)
print(f"Q_oss {q*1e9:.2f} nC, E_oss {e*1e6:.3f} uJ, Q_oss V {q*400*1e6:.2f} uJ, 2 E_oss {2*e*1e6:.2f} uJ")
s = ngspice("coss_loss.cir")
print({k: s[k] for k in ("ediss", "elin")})
v = np.linspace(0, 400, 400)
plt.figure()
plt.semilogy(v, ls.coss_junction(2e-9, 2, 0.9, v) * 1e12)
plt.xlabel("v_DS (V)"); plt.ylabel("C_oss (pF)"); plt.title("Most of the charge goes in at low voltage"); plt.show()

Q_oss 27.98 nC, E_oss 2.421 uJ, Q_oss V 11.19 uJ, 2 E_oss 4.84 uJ
{'ediss': 1.1199e-05, 'elin': 1.11886e-05}


## Part 2 · ZVS transitions

*Predict* the time for 2 A to swing a 400 V half bridge with C_o(tr) = 88 pF (GaN) and 1212 pF (superjunction).

In [3]:
for name, c in (("GaN", 88e-12), ("superjunction", 1212e-12)):
    print(f"{name}: {ls.zvs_transition_time(c*400, 2)*1e9:.1f} ns; current for ZVS in 50 ns {ls.zvs_min_current(c*400, 50e-9):.2f} A")
s = ngspice("zvs.cir")
print({k: s[k] for k in ("ta", "tb")})

GaN: 35.2 ns; current for ZVS in 50 ns 1.41 A
superjunction: 484.8 ns; current for ZVS in 50 ns 19.39 A
{'ta': 3.515482e-08, 'tb': 4.841906e-07}


## Part 3 · Silicon against GaN across frequency

*Predict* the frequency at which each technology's switch losses reach 2% of 240 W in Example 22-06.6's buck.

In [4]:
f = np.linspace(50e3, 2e6, 400)


def budget(f, r, qg, vd, qoss, qrr, t, vsd, vin=48.0, io=20.0):
    return (400 * r + ls.gate_drive(2 * qg, vd, f) + ls.coss_hard_turn_on(qoss, vin, f) + ls.recovery(qrr, vin, f)
            + ls.overlap(vin, io, t, t, f) + ls.dead_time(vsd, io, 20e-9, f))


si = budget(f, 1.8e-3, 73e-9, 10, 135e-9, 70e-9, 8e-9, 0.8)
gan = budget(f, 2.4e-3, 11e-9, 5, 43e-9, 0.0, 3e-9, 2.5)
plt.figure()
plt.plot(f / 1e6, si, label="100 V silicon"); plt.plot(f / 1e6, gan, label="100 V GaN")
plt.axhline(4.8, color="0.3", ls="--"); plt.xlabel("f_s (MHz)"); plt.ylabel("switch loss (W)"); plt.legend(); plt.show()
for name, p in (("Si", si), ("GaN", gan)):
    print(f"{name}: 2% of 240 W reached at {np.interp(4.8, p, f)/1e3:.0f} kHz")

Si: 2% of 240 W reached at 208 kHz
GaN: 2% of 240 W reached at 544 kHz


## Part 4 · The optimal device size

*Predict* the on-resistance that minimizes conduction plus gate loss for 17.32 A RMS at 1 MHz, for the silicon figure of merit.

In [5]:
i, fom, vd, fs = 20 * math.sqrt(0.75), 1.8e-3 * 73e-9, 10, 1e6
r = np.logspace(-4, -2, 400)
cond, gate = i ** 2 * r, fom * vd * fs / r
plt.figure()
plt.loglog(r * 1e3, cond, label="conduction"); plt.loglog(r * 1e3, gate, label="gate"); plt.loglog(r * 1e3, cond + gate, label="total")
plt.xlabel("R_DS(on) (mOhm)"); plt.ylabel("loss (W)"); plt.legend(); plt.show()
print(f"optimum {ls.optimal_resistance(fom, vd, fs, i)*1e3:.2f} mOhm, minimum loss {ls.minimum_loss(fom, vd, fs, i):.2f} W")

optimum 2.09 mOhm, minimum loss 1.26 W


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.